Un MLP è una rete neurale multistrato

Implementazione in Python

Implementazione con Pytorch

In [2]:
# Definizione dell'architettura

import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader


class MLP(nn.Module):
    """
    Rete neurale multistrato (MLP) per classificazione multiclasse.
    Parametri
    ----------
    input_dim   : int   – dimensionalità dello spazio delle feature
    hidden_dim  : list  – lista con il numero di neuroni per strato nascosto
    output_dim  : int   – numero di classi
    dropout     : float – tasso di dropout (regolarizzazione)
    """

    def __init__(self, input_dim, hidden_dim, output_dim, dropout=0.3):
        super(MLP, self).__init__()
        layers = []
        prev_dim = input_dim

        for h in hidden_dim:
            layers.append(nn.Linear(prev_dim, h))
            layers.append(nn.ReLU())
            layers.append(nn.Dropout(p=dropout))
            prev_dim = h

        layers.append(nn.Linear(prev_dim, output_dim))
        # La softmax è inclusa nella CrossEntropyLoss di PyTorch
        self.network = nn.Sequential(*layers)

    def forward(self, x):
        return self.network(x)

Addestramento

In [ ]:
def train_epoch(model, loader, optimizer, criterion, device):
    model.train()
    total_loss = 0.0

    for X_batch, y_batch in loader:
        X_batch, y_batch = X_batch.to(device), y_batch.to(device)
        optimizer.zero_grad()               # Azzeramento dei gradienti accumulati
        logits = model(X_batch)             # Forward pass
        loss = criterion(logits, y_batch)   # Calcolo della loss
        loss.backward()                     # Backward pass (propagation)
        optimizer.step()                    # Aggiornamento dei parametri

        total_loss += loss.item() * X_batch.size(0)

    return total_loss / len(loader.dataset)


# Configurazione del training
device      =   torch.device("cuda" if torch.cuda.is_available() else "cpu")
model       =   MLP(input_dim=784, hidden_dim=[256, 128] , output_dim=10).to(device)
optimizer   =   optim.Adam(model.parameters(), lr=1e-3, weight_decay=1e-4)
criterion   =   nn.CrossEntropyLoss()       # Include log-softmax + NLL

NUM_EPOCHS = 30
for epoch in range(NUM_EPOCHS + 1):
    loss_tr = train_epoch(model, train_loader, optimizer, criterion, device)
    if epoch % 5 == 0:
        print(f"Epoch {epoch:3d}  | Train Loss: {loss_tr:.4f}")

Valutazione

In [ ]:
def evaluate(model, loader, device):
    model.eval()
    correct, total = 0,0

    with torch.no_grad():
        for X_batch, y_batch in loader:
            X_batch = X_batch.to(device)
            preds   = model(X_batch).argmax(dim=1).cpu()
            correct += (preds == y_batch).sum().item()
            total   += y_batch.size(0)

    return correct/total

acc = evaluate(model, test_loader, device)
print(f"Test Accuracy: {acc: .4f}%")

Implementazione con scikit-learn

In [ ]:
from sklearn.neural_network import MLPClassifier
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.metrics import classification_report

pipeline = Pipeline([
    ("scaler", StandardScaler()),
    ("mlp", MLPClassifier(
        hidden_layer_sizes=(256, 128),
        activation="relu",
        solver="adam",
        alpha=1e-4,
        learning_rate_init=1e-3,
        max_iter=200,
        random_state=42,
        early_stopping=True,
        validation_fraction=0.1,
        verbose=True
    ))
])

pipeline.fit(X_train, y_train)
y_pred = pipeline.predict(X_test)
print(classification_report(y_test, y_pred))